# CPU Cluster: Parallel Training with MLflow Child Runs

This notebook trains **90 ML models** in parallel on a CPU cluster:
- Logistic Regression (15 models)
- LinearSVC (10 models) - fast linear SVM with O(n) complexity
- Random Forest (20 models)
- XGBoost (20 models)
- LightGBM Gradient Boosting (15 models)
- Naive Bayes (10 models)

**Key Feature**: Each Ray task logs directly to MLflow as a child run under a parent run.

## Recommended Cluster Configuration

| Setting | Recommendation |
|---------|----------------|
| **Runtime** | Machine Learning Runtime 17.3 LTS or above |
| **Workers** | 8 workers |
| **Cores per node** | 32 cores (256 total cores) |
| **Autoscaling** | Disabled |
| **Access mode** | Dedicated (formerly single user) or No isolation shared |
| **Photon Acceleration** | Disabled (Photon boosts Apache Spark workloads; not all ML workloads will see an improvement) |

**When to use storage or memory optimized compute:**
- **Storage optimized**: Recommended when working with large datasets that require significant disk I/O or shuffle operations
- **Memory optimized**: Recommended when models require large in-memory datasets, feature caching, or when encountering out-of-memory errors

**Note**: Run this notebook on the CPU cluster.

## 1. Setup and Imports

Import Ray, scikit-learn, XGBoost, LightGBM, Optuna, and MLflow for distributed model training.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from datetime import datetime
import json
import time
import os

# Ray imports
import ray

# Scikit-learn models and utilities
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# XGBoost
import xgboost as xgb

# LightGBM
import lightgbm as lgb

# Optuna for Bayesian optimization
import optuna
from optuna.samplers import TPESampler

# PySpark
from pyspark.sql import functions as F

# MLflow for model logging and registry
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from mlflow.models.signature import infer_signature
from mlflow.utils.databricks_utils import get_databricks_env_vars

# Suppress Optuna logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("All imports successful!")
print(f"Notebook type: CPU CLUSTER with MLflow Child Runs")

## 2. Configuration

Define catalog paths, cluster topology, model distribution, and MLflow experiment settings.

In [ ]:
catalog = "ryuta"
schema = "ray"
experiment_name = "/Users/ryuta.yoshimatsu@databricks.com/ray_cpu_model_training"


print(f"Running with parameters:")
print(f"  Catalog: {catalog}")
print(f"  Schema: {schema}")
print(f"  Experiment: {experiment_name}")

# Configuration
CONFIG = {
    'catalog': catalog,
    'schema': schema,
    'table_name': f'{catalog}.{schema}.synthetic_data',
    'results_table': f'{catalog}.{schema}.model_training_results',
    'test_size': 0.2,
    'random_state': 42,
    'n_trials_per_model': 10,  # Bayesian optimization trials
    
    # Cluster configuration
    'cluster_type': 'cpu',
    'n_workers': 8,
    'cores_per_head_node': 0,
    'cores_per_node': 32,
    'total_cores': 256,
    
    # Model distribution (CPU models only)
    'model_distribution': {
        'logistic_regression': 15,
        'svm': 10,
        'random_forest': 20,
        'xgboost': 20,
        'lgbm': 15,
        'naive_bayes': 10
    },
    
    # Model ID offset (CPU models: 0-89)
    'model_id_start': 0,
    
    # MLflow Unity Catalog model registry
    'model_registry_path': f'{catalog}.{schema}',  # catalog.schema
    'experiment_name': experiment_name
}

CONFIG['n_models_total'] = sum(CONFIG['model_distribution'].values())

# Set up MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CONFIG['experiment_name'])

# Get Databricks credentials for Ray workers
# These will be passed to each Ray task to enable MLflow logging
mlflow_db_creds = get_databricks_env_vars("databricks")

print("Configuration loaded successfully!")
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'model_distribution'}, indent=2))
print(f"Model distribution: {CONFIG['model_distribution']}")
print(f"\nMLflow registry URI: {mlflow.get_registry_uri()}")
print(f"MLflow experiment: {CONFIG['experiment_name']}")
print(f"\nDatabricks credentials captured for Ray workers: {list(mlflow_db_creds.keys())}")

## 3. Load Data from Delta Table

Load the synthetic dataset from Delta Lake and split into training and test sets.

In [ ]:
# Load data from Delta table
print(f"Loading data from {CONFIG['table_name']}...")
df_spark = spark.table(CONFIG['table_name'])

print(f"Total rows: {df_spark.count()}")

# Convert to pandas
df = df_spark.toPandas()
print(f"Data loaded successfully! Shape: {df.shape}")

# Prepare features and labels
feature_columns = [col for col in df.columns if col.startswith('feature_')]
X = df[feature_columns].values
y = df['label'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=CONFIG['test_size'], 
    random_state=CONFIG['random_state'],
    stratify=y
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 4. Feature Subset Selection

Generate diverse feature subsets using distributed F-score and mutual information computation.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

def generate_feature_subsets_distributed(n_features, n_subsets, X_train_array, y_train_array, spark_session):
    """
    Generate diverse feature subsets using distributed computation via Pandas UDF.
    Feature importance scores (F-score and MI) are computed in parallel across the cluster.
    """
    np.random.seed(CONFIG['random_state'])
    random_state = CONFIG['random_state']
    
    print("Computing feature importance scores using distributed Pandas UDF...")
    start_time = time.time()
    
    # Broadcast training data to all workers
    X_train_bc = spark_session.sparkContext.broadcast(X_train_array)
    y_train_bc = spark_session.sparkContext.broadcast(y_train_array)
    
    # Create a DataFrame with feature indices
    feature_indices_df = spark_session.createDataFrame(
        [(i,) for i in range(n_features)],
        schema=StructType([StructField("feature_idx", IntegerType(), False)])
    )
    
    # Define the output schema for feature scores
    output_schema = StructType([
        StructField("feature_idx", IntegerType(), False),
        StructField("f_score", DoubleType(), False),
        StructField("mi_score", DoubleType(), False)
    ])
    
    def compute_feature_scores(iterator):
        """Compute F-score and MI score for each feature in the partition"""
        # Access broadcast variables
        X_train_local = X_train_bc.value
        y_train_local = y_train_bc.value
        
        for pdf in iterator:
            results = []
            for idx in pdf['feature_idx'].values:
                # Extract single feature column
                feature_values = X_train_local[:, idx].reshape(-1, 1)
                
                # Compute F-score (ANOVA F-value)
                f_score_val, _ = f_classif(feature_values, y_train_local)
                
                # Compute Mutual Information score
                mi_score_val = mutual_info_classif(
                    feature_values, y_train_local,
                    random_state=random_state
                )
                
                results.append({
                    'feature_idx': int(idx),
                    'f_score': float(f_score_val[0]) if not np.isnan(f_score_val[0]) else 0.0,
                    'mi_score': float(mi_score_val[0]) if not np.isnan(mi_score_val[0]) else 0.0
                })
            
            yield pd.DataFrame(results)
    
    # Repartition to distribute work across cluster and compute scores
    num_partitions = min(n_features, 100)  # Use reasonable number of partitions
    scores_df = (
        feature_indices_df
        .repartition(num_partitions)
        .mapInPandas(compute_feature_scores, schema=output_schema)
    )
    
    # Collect and sort scores
    scores_pandas = scores_df.orderBy("feature_idx").toPandas()
    f_scores = scores_pandas['f_score'].values
    mi_scores = scores_pandas['mi_score'].values
    
    # Clean up broadcast variables
    X_train_bc.unpersist()
    y_train_bc.unpersist()
    
    compute_time = time.time() - start_time
    print(f"Feature importance computation completed in {compute_time:.2f}s")
    
    # Sort features by scores (descending)
    top_f_features = np.argsort(f_scores)[::-1]
    top_mi_features = np.argsort(mi_scores)[::-1]
    
    # Generate subsets using various strategies
    subsets = []
    for i in range(n_subsets):
        subset_size = np.random.randint(20, n_features + 1)
        strategy = i % 7
        
        if strategy == 0:
            subset = list(range(n_features))
        elif strategy == 1:
            subset = sorted([int(x) for x in np.random.choice(n_features, subset_size, replace=False)])
        elif strategy == 2:
            subset = sorted([int(x) for x in top_f_features[:subset_size]])
        elif strategy == 3:
            subset = sorted([int(x) for x in top_mi_features[:subset_size]])
        elif strategy == 4:
            start = int(np.random.randint(0, n_features - subset_size + 1))
            subset = list(range(start, start + subset_size))
        elif strategy == 5:
            n_top = subset_size // 2
            top_features = [int(x) for x in top_f_features[:n_top]]
            remaining = [f for f in range(n_features) if f not in top_features]
            random_features = [int(x) for x in np.random.choice(remaining, subset_size - n_top, replace=False)]
            subset = sorted(top_features + random_features)
        else:
            n_top = subset_size // 2
            top_features = [int(x) for x in top_mi_features[:n_top]]
            remaining = [f for f in range(n_features) if f not in top_features]
            random_features = [int(x) for x in np.random.choice(remaining, subset_size - n_top, replace=False)]
            subset = sorted(top_features + random_features)
        
        subsets.append({
            'feature_indices': subset,
            'n_features': len(subset),
            'strategy': ['all', 'random', 'top_f', 'top_mi', 'block', 'f_random_mix', 'mi_random_mix'][strategy]
        })
    
    return subsets

# Generate feature subsets using distributed computation
n_features = X.shape[1]
feature_subsets = generate_feature_subsets_distributed(
    n_features, 
    CONFIG['n_models_total'], 
    X_train, 
    y_train, 
    spark
)

print(f"Generated {len(feature_subsets)} feature subsets")

## 5. Hyperparameter Spaces

Define search ranges for each model type including regularization, tree depth, and learning rates.

In [ ]:
# Hyperparameter spaces for CPU models
# Tuple formats:
#   (min, max) -> integer range
#   (min, max, 'float') -> float range  
#   (min, max, 'log') -> log-scale float range
HYPERPARAMETER_SPACES = {
    'logistic_regression': {
        'C': (1e-4, 1e2, 'log'),
        'penalty': ['l2'],
        'solver': ['lbfgs', 'saga'],
        'max_iter': [500]
    },
    
    # Uses CalibratedClassifierCV for probability estimates
    'svm': {
        'C': (1e-3, 1e2, 'log'),
        'penalty': ['l2'],
        'loss': ['squared_hinge'],
        'max_iter': [2000],
        'dual': ['auto']
    },
    
    'random_forest': {
        'n_estimators': (50, 300),
        'max_depth': (5, 30),
        'min_samples_split': (2, 20),
        'min_samples_leaf': (1, 10),
        'max_features': ['sqrt', 'log2', None]
    },
    
    'xgboost': {
        'n_estimators': (50, 300),
        'max_depth': (3, 15),
        'learning_rate': (1e-3, 0.3, 'log'),
        'subsample': (0.5, 1.0, 'float'),
        'colsample_bytree': (0.5, 1.0, 'float'),
        'min_child_weight': (1, 10),
        'gamma': (0, 5)
    },
    
    'lgbm': {
        'n_estimators': (50, 300),
        'max_depth': (3, 15),
        'learning_rate': (1e-3, 0.3, 'log'),
        'subsample': (0.5, 1.0, 'float'),
        'colsample_bytree': (0.5, 1.0, 'float'),
        'num_leaves': (20, 100),
        'min_child_samples': (5, 50),
        'reg_alpha': (1e-8, 1.0, 'log'),
        'reg_lambda': (1e-8, 1.0, 'log')
    },
    
    'naive_bayes': {
        'var_smoothing': (1e-11, 1e-7, 'log')
    }
}

print("Hyperparameter spaces defined!")

## 6. Training Functions

Implement training functions for scikit-learn, XGBoost, and LightGBM models with standardized metrics.

In [ ]:
def train_sklearn_model(model_type, hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train a scikit-learn model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    if model_type == 'logistic_regression':
        model = LogisticRegression(**hyperparams, random_state=random_state)
    elif model_type == 'svm':
        # Use LinearSVC wrapped with CalibratedClassifierCV for probability estimates
        base_svm = LinearSVC(**hyperparams, random_state=random_state)
        model = CalibratedClassifierCV(base_svm, cv=3)
    elif model_type == 'random_forest':
        # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
        model = RandomForestClassifier(**hyperparams, random_state=random_state, n_jobs=n_cpus)
    elif model_type == 'naive_bayes':
        model = GaussianNB(**hyperparams)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # Scale features
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Train
    model.fit(X_tr_scaled, y_tr)
    
    # Predict
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics


def train_xgboost_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train XGBoost model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
    model = xgb.XGBClassifier(
        **hyperparams,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=random_state,
        n_jobs=n_cpus,
        use_label_encoder=False
    )
    
    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)], verbose=False)
    
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics


def train_lgbm_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, return_model=False, n_cpus=1):
    """Train LightGBM model
    
    Args:
        n_cpus: Number of CPUs to use for parallelization (default=1, use explicit value to avoid resource contention)
    """
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    # Use explicit n_cpus instead of -1 to avoid resource contention in Ray tasks
    model = lgb.LGBMClassifier(
        **hyperparams,
        random_state=random_state,
        n_jobs=n_cpus,
        verbose=-1  # Suppress LightGBM output
    )
    
    model.fit(X_tr_scaled, y_tr, eval_set=[(X_val_scaled, y_val)])
    
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_pred_proba),
        'f1': f1_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred)
    }
    
    if return_model:
        return metrics, model, scaler
    return metrics

print("Training functions defined!")

## 7. Bayesian Optimization

Use Optuna's TPE sampler to efficiently search hyperparameter space and maximize ROC AUC.

In [ ]:
def suggest_hyperparameters(trial, model_type, hyperparameter_spaces):
    """Suggest hyperparameters using Optuna
    
    Tuple formats:
      (min, max) -> integer range
      (min, max, 'float') -> float range
      (min, max, 'log') -> log-scale float range
    """
    space = hyperparameter_spaces[model_type]
    hyperparams = {}
    
    for param_name, param_config in space.items():
        if isinstance(param_config, tuple):
            if len(param_config) == 3:
                if param_config[2] == 'log':
                    hyperparams[param_name] = trial.suggest_float(
                        param_name, param_config[0], param_config[1], log=True
                    )
                elif param_config[2] == 'float':
                    hyperparams[param_name] = trial.suggest_float(
                        param_name, param_config[0], param_config[1]
                    )
            else:
                hyperparams[param_name] = trial.suggest_int(
                    param_name, param_config[0], param_config[1]
                )
        elif isinstance(param_config, list):
            hyperparams[param_name] = trial.suggest_categorical(param_name, param_config)
    
    return hyperparams


def optimize_hyperparameters(model_type, X_tr, y_tr, X_val, y_val, hyperparameter_spaces, random_state, n_trials=20, n_cpus=1):
    """Optimize hyperparameters using Bayesian optimization
    
    Args:
        n_cpus: Number of CPUs to use for model training (passed to training functions)
    """
    
    def objective(trial):
        hyperparams = suggest_hyperparameters(trial, model_type, hyperparameter_spaces)
        
        try:
            if model_type == 'xgboost':
                metrics = train_xgboost_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
            elif model_type == 'lgbm':
                metrics = train_lgbm_model(hyperparams, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
            else:
                metrics = train_sklearn_model(model_type, hyperparams, X_tr, y_tr, X_val, y_val, random_state, n_cpus=n_cpus)
            
            return metrics['roc_auc']
        except Exception:
            return 0.5
    
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=random_state)
    )
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    
    return study.best_params, study.best_value

print("Bayesian optimization functions defined!")

## 8. Ray Remote Training Function with MLflow Child Runs

Define a Ray remote function that trains a model with Bayesian HPO and logs results as an MLflow child run.

In [ ]:
def convert_numpy_types(obj):
    """Convert numpy types to native Python for JSON serialization"""
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


@ray.remote
def train_model_with_mlflow_child_run(
    model_id,
    model_type,
    feature_subset,
    X_train_full,
    y_train_full,
    X_test_full,
    y_test_full,
    parent_run_id,
    mlflow_db_creds,
    experiment_name,
    model_registry_path,
    hyperparameter_spaces,
    random_state,
    n_trials=20,
    n_cpus=1
):
    """
    Ray remote function to train a single model with Bayesian hyperparameter optimization.
    
    Logs parameters, metrics, and models directly to MLflow as a child run.
    This establishes a hierarchical parent-child relationship in MLflow.
    """
    import os
    import time
    import json
    import numpy as np
    import mlflow
    import mlflow.sklearn
    import mlflow.xgboost
    import mlflow.lightgbm
    from mlflow.models.signature import infer_signature
    from sklearn.model_selection import train_test_split
    
    start_time = time.time()
    
    try:
        # Set MLflow credentials within the Ray task
        os.environ.update(mlflow_db_creds)
        
        # Set MLflow to use Unity Catalog and the correct experiment
        mlflow.set_registry_uri("databricks-uc")
        mlflow.set_experiment(experiment_name)
        
        # Extract feature subset
        feature_indices = feature_subset['feature_indices']
        X_train = X_train_full[:, feature_indices]
        X_test = X_test_full[:, feature_indices]
        
        # Split for validation
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train_full,
            test_size=0.2,
            random_state=random_state,
            stratify=y_train_full
        )
        
        # Optimize hyperparameters (pass n_cpus for proper resource usage)
        best_hyperparams, best_val_score = optimize_hyperparameters(
            model_type, X_tr, y_tr, X_val, y_val, hyperparameter_spaces, random_state, n_trials, n_cpus=n_cpus
        )
        
        # Train final model on full training set (with return_model=True)
        if model_type == 'xgboost':
            test_metrics, final_model, scaler = train_xgboost_model(
                best_hyperparams, X_train, y_train_full, X_test, y_test_full, random_state, return_model=True, n_cpus=n_cpus
            )
        elif model_type == 'lgbm':
            test_metrics, final_model, scaler = train_lgbm_model(
                best_hyperparams, X_train, y_train_full, X_test, y_test_full, random_state, return_model=True, n_cpus=n_cpus
            )
        else:
            test_metrics, final_model, scaler = train_sklearn_model(
                model_type, best_hyperparams, X_train, y_train_full, X_test, y_test_full, random_state, return_model=True, n_cpus=n_cpus
            )
        
        training_time = time.time() - start_time
        
        # Log to MLflow as a child run
        registered_model_name = f"{model_registry_path}.cpu_model_{model_id}_child"
        
        # Create a nested child run associated with the parent run_id
        with mlflow.start_run(run_name=f"cpu_model_{model_id}_{model_type}", parent_run_id=parent_run_id, nested=True) as child_run:
            # Log parameters
            mlflow.log_params(convert_numpy_types(best_hyperparams))
            mlflow.log_param("model_type", model_type)
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("n_features_used", len(feature_indices))
            mlflow.log_param("feature_strategy", feature_subset['strategy'])
            mlflow.log_param("n_trials", n_trials)
            
            # Log metrics
            mlflow.log_metrics({
                'accuracy': float(test_metrics['accuracy']),
                'roc_auc': float(test_metrics['roc_auc']),
                'f1': float(test_metrics['f1']),
                'precision': float(test_metrics['precision']),
                'recall': float(test_metrics['recall']),
                'best_val_score': float(best_val_score),
                'training_time': float(training_time)
            })
            
            # Log feature indices as artifact
            feature_indices_str = json.dumps(convert_numpy_types(feature_indices))
            mlflow.log_text(feature_indices_str, "feature_indices.json")
            
            # Create model signature
            X_test_scaled = scaler.transform(X_test)
            signature = infer_signature(X_test_scaled, final_model.predict(X_test_scaled))
            
            # Log and register model based on type
            if model_type == 'xgboost':
                mlflow.xgboost.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            elif model_type == 'lgbm':
                mlflow.lightgbm.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            else:
                mlflow.sklearn.log_model(
                    final_model,
                    artifact_path="model",
                    signature=signature,
                    registered_model_name=registered_model_name
                )
            
            child_run_id = child_run.info.run_id
        
        # Return result summary (without model objects - they're already logged to MLflow)
        result = {
            'model_id': model_id,
            'model_type': model_type,
            'cluster_type': 'cpu',
            'n_features_used': len(feature_indices),
            'feature_strategy': feature_subset['strategy'],
            'best_hyperparams': json.dumps(convert_numpy_types(best_hyperparams)),
            'best_val_score': float(best_val_score),
            'accuracy': float(test_metrics['accuracy']),
            'roc_auc': float(test_metrics['roc_auc']),
            'f1': float(test_metrics['f1']),
            'precision': float(test_metrics['precision']),
            'recall': float(test_metrics['recall']),
            'training_time': float(training_time),
            'mlflow_run_id': child_run_id,
            'registered_model_name': registered_model_name,
            'status': 'success'
        }
        
        return result
        
    except Exception as e:
        import traceback
        return {
            'model_id': model_id,
            'model_type': model_type,
            'cluster_type': 'cpu',
            'status': 'failed',
            'error': str(e),
            'traceback': traceback.format_exc(),
            'training_time': time.time() - start_time
        }

print("Ray remote training function with MLflow child runs defined!")

## 9. Initialize Ray

Start a Ray cluster across all Spark worker nodes with 256 total CPU cores for parallel training. For more details on memory allocation for Ray worker nodes, see [`here`](https://docs.databricks.com/aws/en/machine-learning/ray/scale-ray#memory-allocation-for-ray-worker-nodes).

In [ ]:
# Initialize Ray cluster using Databricks utilities
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Shutdown any existing Ray instance
if ray.is_initialized():
    ray.shutdown()

try:
    # Setup Ray cluster spanning all Spark nodes
    # autoscale=False ensures fixed cluster size (no dynamic scaling)
    print("Setting up Ray cluster across all nodes (autoscaling disabled)...")
    setup_ray_cluster(
        min_worker_nodes=CONFIG['n_workers'],  # 8 worker nodes
        max_worker_nodes=CONFIG['n_workers'],  # 8 worker nodes
        num_cpus_head_node=CONFIG['cores_per_head_node'], # 16 CPUs per head node
        num_cpus_worker_node=CONFIG['cores_per_node'],  # 32 CPUs per worker node
        num_gpus_head_node=0,   # 0 GPUs for CPU cluster
        num_gpus_worker_node=0,  # 0 GPUs for CPU cluster
        collect_log_to_path="/Workspace/Users/ryuta.yoshimatsu@databricks.com/ray_logs",
    )
    # Explicitly initialize Ray after cluster setup
    ray.init(
        address='auto', 
        ignore_reinit_error=True, 
        logging_level='ERROR',
    )
    
except Exception as e:
    print(f"Warning: setup_ray_cluster failed with: {e}")
    print("Falling back to standard Ray initialization...")    
    ray.init(
        ignore_reinit_error=True, 
        logging_level='ERROR',
    )

print("\nRay initialized successfully on CPU cluster!")
print(f"Available CPUs: {ray.cluster_resources().get('CPU', 0)}")
print(f"Available GPUs: {ray.cluster_resources().get('GPU', 0)}")
print(f"Expected CPUs: {CONFIG['total_cores']} (8 workers × 32 cores)")

# Verify cluster setup
print(f"\nCluster nodes connected: {len(ray.nodes())}")
print(f"Total cluster resources: {ray.cluster_resources()}")

## 10. Generate Model Configurations

Create 90 model configurations with assigned feature subsets and CPU allocations based on model type.

In [ ]:
# Generate model configurations
model_configs = []
model_id = CONFIG['model_id_start']

for model_type, count in CONFIG['model_distribution'].items():
    for i in range(count):
        feature_subset = feature_subsets[model_id % len(feature_subsets)]
        
        # Determine CPU allocation
        # Models with parallelization support (n_jobs) get 4 CPUs
        if model_type in ['random_forest', 'xgboost', 'lgbm']:
            n_cpus = 4
        else:
            n_cpus = 1
        
        config = {
            'model_id': model_id,
            'model_type': model_type,
            'feature_subset': feature_subset,
            'n_cpus': n_cpus
        }
        
        model_configs.append(config)
        model_id += 1

print(f"Generated {len(model_configs)} CPU model configurations")
print(f"Model IDs: {CONFIG['model_id_start']} to {model_id - 1}")
print(f"\nModel distribution:")
for model_type, count in CONFIG['model_distribution'].items():
    print(f"  {model_type}: {count}")

## 11. Launch Parallel Training with MLflow Parent Run

Create a parent MLflow run and submit all 90 training tasks to Ray, collecting results as they complete.

In [ ]:
# Put data in Ray object store
X_train_ref = ray.put(X_train)
y_train_ref = ray.put(y_train)
X_test_ref = ray.put(X_test)
y_test_ref = ray.put(y_test)

print("Data stored in Ray object store")

This cell implements a **fan-out / fan-in** pattern in three stages:

1. **Fan-out (task submission)** -- Iterates over all 90 model configs and calls `.remote()` to enqueue each as an async Ray task. `.options(num_cpus=...)` tells Ray's scheduler how many cores to reserve (4 for tree-based models with `n_jobs` parallelism, 1 for simpler models). Data is passed by reference (`X_train_ref`, etc.) to avoid redundant copies. MLflow credentials and the parent run ID are passed so each worker can log a child run from a remote node.

2. **Streaming result collection** -- Uses `ray.wait(remaining, num_returns=1)` in a loop to retrieve results **in completion order** (not submission order), printing live progress as each model finishes. This is preferred over `ray.get(futures)` which would block until the slowest task.

3. **Parent-run aggregation** -- After all tasks complete, computes summary statistics (mean/best ROC AUC, success/failure counts, total wall time) and logs them to the **parent** MLflow run, creating a two-level hierarchy: parent holds the overview, each child run holds per-model details.

In [ ]:
# Start parent run on the main driver process
# All Ray tasks will create child runs under this parent

print(f"\nStarting MLflow parent run and launching {len(model_configs)} training jobs...\n")
print("="*80)
start_time = time.time()

# Create the parent MLflow run
with mlflow.start_run(run_name="cpu_parallel_training_parent") as parent_run:
    parent_run_id = parent_run.info.run_id
    print(f"Parent MLflow run ID: {parent_run_id}")
    
    # Log parent run metadata
    mlflow.log_params({
        'n_models': len(model_configs),
        'cluster_type': CONFIG['cluster_type'],
        'n_workers': CONFIG['n_workers'],
        'total_cores': CONFIG['total_cores'],
        'n_trials_per_model': CONFIG['n_trials_per_model']
    })
    
    # Launch all training jobs as Ray tasks
    # Each task will create a child run under the parent
    futures = []
    for config in model_configs:
        remote_fn = train_model_with_mlflow_child_run.options(num_cpus=config['n_cpus'])
        
        future = remote_fn.remote(
            model_id=config['model_id'],
            model_type=config['model_type'],
            feature_subset=config['feature_subset'],
            X_train_full=X_train_ref,
            y_train_full=y_train_ref,
            X_test_full=X_test_ref,
            y_test_full=y_test_ref,
            parent_run_id=parent_run_id,
            mlflow_db_creds=mlflow_db_creds,
            experiment_name=CONFIG['experiment_name'],
            model_registry_path=CONFIG['model_registry_path'],
            hyperparameter_spaces=HYPERPARAMETER_SPACES,
            random_state=CONFIG['random_state'],
            n_trials=CONFIG['n_trials_per_model'],
            n_cpus=config['n_cpus']
        )
        
        futures.append(future)
    
    print(f"All {len(futures)} jobs submitted. Waiting for results...\n")
    
    # Collect results from Ray workers
    results = []
    completed = 0
    remaining_futures = futures.copy()
    
    print("Collecting training results from workers (each logging to MLflow as child run)...")
    
    while remaining_futures:
        ready_futures, remaining_futures = ray.wait(remaining_futures, num_returns=1)
        
        for future in ready_futures:
            result = ray.get(future)
            results.append(result)
            completed += 1
            
            if result['status'] == 'success':
                print(f"[{completed}/{len(futures)}] Model {result['model_id']} ({result['model_type']}) - "
                      f"ROC AUC: {result['roc_auc']:.4f} ({result['training_time']:.1f}s) -> MLflow child run logged")
            else:
                print(f"[{completed}/{len(futures)}] Model {result['model_id']} FAILED: {result.get('error', 'Unknown')}")
    
    training_time = time.time() - start_time
    
    # Log summary metrics to parent run
    successful_results = [r for r in results if r['status'] == 'success']
    failed_results = [r for r in results if r['status'] != 'success']
    
    if successful_results:
        roc_aucs = [r['roc_auc'] for r in successful_results]
        accuracies = [r['accuracy'] for r in successful_results]
        
        mlflow.log_metrics({
            'total_training_time': training_time,
            'n_successful': len(successful_results),
            'n_failed': len(failed_results),
            'mean_roc_auc': np.mean(roc_aucs),
            'best_roc_auc': np.max(roc_aucs),
            'mean_accuracy': np.mean(accuracies),
            'best_accuracy': np.max(accuracies)
        })

print(f"\n{'='*80}")
print(f"ALL TRAINING JOBS COMPLETE")
print(f"{'='*80}")
print(f"Total time: {training_time:.2f}s ({training_time/60:.2f} minutes)")
print(f"Successful: {len(successful_results)}")
print(f"Failed: {len(failed_results)}")
print(f"\nParent MLflow run ID: {parent_run_id}")
print(f"All child runs logged under: {CONFIG['experiment_name']}")

In [ ]:
# Shutdown Ray cluster using shutdown_ray_cluster() for clean Spark integration
from ray.util.spark import shutdown_ray_cluster
import ray

print("\nShutting down Ray cluster...")
try:
    shutdown_ray_cluster()
    print("Ray cluster shut down successfully!")
except Exception as e:
    print(f"Warning: Ray cluster shutdown encountered an error (non-critical): {e}")
    print("Ray cluster will be automatically cleaned up when notebook is detached.")

# Also shutdown Ray client
try:
    ray.shutdown()
    print("Ray client shut down successfully!")
except Exception as e:
    print(f"Warning: Ray client shutdown encountered an error: {e}")

## Summary

This notebook trained **90 traditional ML models** using **MLflow child runs**:

**Key Architecture Changes:**
- Parent MLflow run created on the driver process
- Each Ray task creates a child run under the parent
- MLflow credentials passed to Ray workers via `get_databricks_env_vars`
- Parameters, metrics, and models logged directly from Ray workers

**Benefits:**
- Hierarchical organization of experiments (parent-child structure)
- Real-time logging during training (not post-collection)
- Better visibility into individual model training progress
- Models registered to Unity Catalog: `{catalog}.{schema}.cpu_model_*_child`

**Next Steps:**
1. View the parent run in MLflow to see all child runs
2. Compare model performance across child runs
3. Run the GPU notebook for PyTorch models